# 面试问题：什么时候用固定 Workflow，什么时候用 Agent？Sequential、Routing、Parallel、Orchestrator 和 Evaluator-Optimizer 怎样选？

**一句话回答**：路径可预先枚举、风险高或要求确定性时优先 Workflow；只有子任务无法事先确定、环境反馈会改变下一步且收益覆盖额外成本时才升级为 Agent。Sequential 适合稳定阶段链，Routing 适合互斥类别，Parallel 适合独立子任务，Orchestrator-Workers 适合动态分解，Evaluator-Optimizer 适合有明确 rubric 且迭代能验证提升的任务。

本 Notebook 用纯 Python 实现五种控制流、统一预算和选择器，重点展示停止条件与失败传播，而不是把多个模型调用都称作 Agent。

In [ ]:
from dataclasses import dataclass
import hashlib, json, math
import numpy as np

SEED115=11501; rng115=np.random.default_rng(SEED115)
PATTERNS115=("sequential","routing","parallel","orchestrator_workers","evaluator_optimizer","agent")
assert len(PATTERNS115)==6 and len(set(PATTERNS115))==6
assert SEED115==11501
assert hashlib.sha256(b"workflow").hexdigest()!=hashlib.sha256(b"agent").hexdigest()

## 1. 用任务属性而不是潮流决定复杂度

先问：步骤是否已知、子任务是否独立、是否有确定 rubric、环境是否动态、动作风险、可接受成本/延迟。Agent 的自由度带来适应性，也带来更多模型轮次、错误累积和不可预测路径；复杂度必须有可测增益。

In [ ]:
@dataclass(frozen=True)
class Task115:
    name:str; known_steps:bool; independent_parts:bool=False; distinct_routes:bool=False; clear_rubric:bool=False; dynamic_environment:bool=False; high_risk:bool=False
    def __post_init__(self):
        if not self.name: raise ValueError("task_contract")
invoice115=Task115("发票抽取",True,distinct_routes=True,high_risk=True)
research115=Task115("开放研究",False,independent_parts=True,dynamic_environment=True)
translation115=Task115("高质量翻译",True,clear_rubric=True)
assert invoice115.known_steps and research115.dynamic_environment
assert translation115.clear_rubric and not translation115.high_risk
try: Task115("",True); raise AssertionError("empty task accepted")
except ValueError as e: assert str(e)=="task_contract"

## 2. Sequential：阶段稳定、错误可逐层定位

每一步输出满足下一步 schema，失败立即停止或进入明确补偿。它适合“解析→检索→生成→校验”这类固定路径。下面用函数列表实现，并为每步保留输入输出摘要；真实系统还要设置每步 timeout 与版本。

In [ ]:
def sequential115(value,steps):
    trace=[]
    for name,fn in steps:
        before=value; value=fn(value); trace.append({"step":name,"input":before,"output":value})
    return value,trace
steps115=[("strip",lambda s:s.strip()),("lower",lambda s:s.lower()),("tokenize",lambda s:s.split())]
seq_out115,seq_trace115=sequential115("  RAG Agent  ",steps115)
assert seq_out115==["rag","agent"]
assert [x["step"] for x in seq_trace115]==["strip","lower","tokenize"]
assert seq_trace115[0]["input"]=="  RAG Agent  "

## 3. Routing：先分类，再进入专用路径

类别应互斥且路由可高置信识别，例如退款、技术支持、一般问答。低置信走通用路径或人工，不能强行分配。路由器和每条分支分别评测，同时监控类别分布漂移。

In [ ]:
ROUTES115={"refund":lambda q:"退款流程", "technical":lambda q:"技术排查", "general":lambda q:"通用回答"}
def classify115(query):
    if "退款" in query: return "refund",.98
    if "报错" in query: return "technical",.91
    return "general",.55
def route115(query,min_conf=.8):
    label,conf=classify115(query); chosen=label if conf>=min_conf else "general"; return chosen,ROUTES115[chosen](query),conf
assert route115("我要退款")[0:2]==("refund","退款流程")
assert route115("系统报错")[0]=="technical"
assert route115("含糊请求")[0]=="general" and route115("含糊请求")[2]<.8

## 4. Parallel：只并行真正独立的工作

Sectioning 把不同维度交给独立 worker，Voting 对同题做多次独立判断。并行能降低墙钟时间或增加覆盖，但总 token 成本通常增加；共享可变状态、存在依赖或无确定聚合规则时不应并行。

In [ ]:
def parallel_sections115(text,workers):
    outputs={name:fn(text) for name,fn in workers.items()}; return {k:outputs[k] for k in sorted(outputs)}
workers115={"correctness":lambda s:int("4" in s),"safety":lambda s:int("密码" not in s),"style":lambda s:int(len(s)<30)}
parallel115=parallel_sections115("答案是4",workers115)
assert parallel115=={"correctness":1,"safety":1,"style":1}
assert list(parallel115)==sorted(workers115)
assert sum(parallel115.values())==3

## 5. Orchestrator-Workers：子任务由输入动态决定

Orchestrator 只生成有 schema 的任务清单，宿主校验依赖、预算和权限后再调度 worker。它适合跨文件修改或多源研究；若拆分规则固定，普通 parallel workflow 更简单、更可测。聚合器要处理重复、冲突和缺失结果。

In [ ]:
def decompose115(request):
    tasks=[]
    if "价格" in request: tasks.append({"id":"price","worker":"market"})
    if "风险" in request: tasks.append({"id":"risk","worker":"risk"})
    if "总结" in request: tasks.append({"id":"summary","worker":"writer"})
    return tasks
WORKERS115={"market":lambda _:"价格=10","risk":lambda _:"风险=低","writer":lambda _:"待聚合"}
plan115=decompose115("调查价格与风险并总结"); results115={t["id"]:WORKERS115[t["worker"]](None) for t in plan115}
assert [t["id"] for t in plan115]==["price","risk","summary"]
assert results115["price"]=="价格=10" and results115["risk"]=="风险=低"
assert set(t["worker"] for t in plan115)<=set(WORKERS115)

## 6. Evaluator-Optimizer：必须有可验证的进步和停止条件

Evaluator 输出结构化缺陷，Optimizer 只修这些缺陷；每轮保留 best-so-far。达到 rubric、改进小于阈值、重复草稿、预算耗尽或 evaluator 不稳定时停止。没有清晰 rubric 的开放审美任务可能越改越差。

In [ ]:
REQUIRED115={"结论","证据","限制"}
def evaluate115(draft): return REQUIRED115-set(draft)
def optimize115(draft,missing): return draft|set(sorted(missing)[:1])
def evaluator_loop115(initial,max_rounds=5):
    draft=set(initial); history=[set(draft)]
    for _ in range(max_rounds):
        missing=evaluate115(draft)
        if not missing: return draft,history,"passed"
        new=optimize115(draft,missing)
        if new==draft: return draft,history,"no_progress"
        draft=new; history.append(set(draft))
    return draft,history,"budget"
final115,hist115,stop115=evaluator_loop115({"结论"})
assert final115==REQUIRED115 and stop115=="passed"
assert len(hist115)==3
assert all(len(a)<len(b) for a,b in zip(hist115,hist115[1:]))

## 7. 选择器从最简单可行方案开始

高风险且路径已知强制 workflow；明显类别用 routing；独立子任务用 parallel；动态分解用 orchestrator；明确 rubric 用 evaluator；只有环境反馈不可预知且步骤未知时才考虑 autonomous agent。组合时仍由外层确定性预算控制。

In [ ]:
def choose_pattern115(task):
    if task.high_risk and task.known_steps: return "sequential"
    if task.distinct_routes: return "routing"
    if task.known_steps and task.independent_parts: return "parallel"
    if not task.known_steps and task.independent_parts: return "orchestrator_workers"
    if task.clear_rubric: return "evaluator_optimizer"
    if task.dynamic_environment: return "agent"
    return "sequential"
assert choose_pattern115(invoice115)=="sequential"
assert choose_pattern115(research115)=="orchestrator_workers"
assert choose_pattern115(translation115)=="evaluator_optimizer"

## 8. 用质量—成本—延迟实验证明复杂度值得

同一黄金集比较 direct prompt、workflow 和 agent，报告 task success、违规率、p95、模型调用数和 token。复杂方案必须有显著质量收益且不越硬 SLO；上线先 shadow，再 canary，并保留降级到简单 workflow 的开关。

In [ ]:
rows115=[{"name":"prompt","quality":.70,"calls":1,"p95":.5},{"name":"workflow","quality":.84,"calls":3,"p95":1.1},{"name":"agent","quality":.87,"calls":8,"p95":3.8}]
feasible115=[r for r in rows115 if r["quality"]>=.82 and r["p95"]<=2]; selected_arch115=min(feasible115,key=lambda r:r["calls"])
manifest115={"schema":1,"selector":"simplest-feasible-v1","patterns":list(PATTERNS115),"quality_floor":.82,"p95_limit":2,"fallback":"sequential"}; digest115=hashlib.sha256(json.dumps(manifest115,sort_keys=True).encode()).hexdigest()
assert selected_arch115["name"]=="workflow"
assert manifest115["fallback"]=="sequential" and len(manifest115["patterns"])==6
assert len(digest115)==64

## 面试总结

最稳的回答是：**先判断步骤可预测性和风险 → sequential / routing / parallel → 动态分解才 orchestrator → 有客观 rubric 才 evaluator loop → 环境驱动且路径未知才 agent → 同黄金集证明复杂度收益**。Agent 是最后一级控制自由度，不是所有 LLM 应用的默认架构。

延伸阅读：[Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)、[ReAct](https://arxiv.org/abs/2210.03629)、[AgentBench](https://arxiv.org/abs/2308.03688)。